In [ ]:
import os, platform, torch

# Определяем рабочую директорию в зависимости от среды
if platform.system() == 'Darwin':
    # macOS (локально)
    REPO_DIR = '/Users/amir/sciml/diffusion_data_assimilation'
else:
    # Linux-сервер
    REPO_DIR = '/home'

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")

In [2]:
import torch
from trainer import TrainingConfig, UNetTrainer
from utils import NpyImageDataset, SequentialIceDataset, channel_normalize, add_noise
from diffusers.optimization import get_cosine_schedule_with_warmup
from model import VideoDiffusionModel

In [3]:
channel_mean = [0.1382167, 0.1816227]
channel_std  = [0.32978467, 0.51380478]

In [ ]:
config = TrainingConfig()

dataset_train = SequentialIceDataset(
    folder="/mnt/sciml/a.sadreev/sea_ice_data/valid",
    transform=lambda x: channel_normalize(x, channel_mean, channel_std),
    num_frames=1,
    stride=1,
    mmap_mode='r',
)

# pin_memory только для CUDA; num_workers=0 на MPS во избежание fork-ошибок
_is_cuda = torch.cuda.is_available()
_is_mps  = torch.backends.mps.is_available()
_num_workers = 0 if _is_mps else 6

train_dataloader = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=config.train_batch_size,
    shuffle=True,
    num_workers=_num_workers,
    pin_memory=_is_cuda,
)

In [ ]:
dataset_valid = NpyImageDataset(
    folder="/mnt/sciml/a.sadreev/sea_ice_data/test",
    transform=lambda x: channel_normalize(x, channel_mean, channel_std),
    preload=False,
    mmap_mode='r',
)

valid_dataloader = torch.utils.data.DataLoader(
    dataset_valid,
    batch_size=config.eval_batch_size,
    shuffle=False,
    num_workers=_num_workers,
    pin_memory=_is_cuda,
)

In [6]:
model = VideoDiffusionModel(
    in_channels=2,         
    dim=64,                 # базовая ширина (64 → 128 → 256 если нужно больше ёмкости)
    dim_mults=(1, 2, 4, 8), # 3 downsampling: 320/8=40 ✓, 256/8=32 ✓
    temporal_compression=(False, False, False, False),
)

print(f'Параметров: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

Non-A100 GPU detected, using math or mem efficient attention if input tensor is on cuda
Параметров: 61.7M


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=config.lr_warmup_steps,
    num_training_steps=(len(train_dataloader) * config.num_epochs),
)

trainer = UNetTrainer(config=config,
                       model=model, 
                       optimizer=optimizer, 
                       data_loader_train=train_dataloader, 
                       data_loader_val=valid_dataloader, 
                       lr_scheduler=lr_scheduler, 
                       add_noise_func=add_noise)

In [8]:
trainer.train_loop()

  0%|          | 0/548 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Epoch   0 | train=0.5201 | val=0.1943 | lr=1.00e-04
  ↳ New best! val_loss=0.194341
  ↳ Samples: checkpoints/run_20260217_171203/samples/epoch_0000.png


  0%|          | 0/548 [00:00<?, ?it/s]

KeyboardInterrupt: 